# D2-04 Spatial matching and fallback logic

This notebook focuses on the geography-matching logic used by `edges` once a method file and an inventory are already in place.


## Learning goals

After this notebook, you should be able to:

- explain why spatial matching is needed in regionalised LCIA
- distinguish direct, aggregate, dynamic, contained, and global fallback mapping
- predict which matching strategy should apply in simple geography cases
- identify the helper methods in `edges` that extend matching beyond direct hits


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7


## 1) Why matching logic is needed

A method file may define CFs for `FR`, `RER`, `RoW`, or `GLO`, while the inventory can contain exchanges located in `FR`, `ES`, `CA-QC`, or other geographies. `edges` therefore needs a transparent sequence of matching rules.


In [ ]:
import json
from pathlib import Path

import pandas as pd

example_2_path = Path('/Users/romain/GitHub/barcelona-regionalized-lcia/tutorials/DAY 2/assets/edges_examples/lcia_example_2.json')
data = json.loads(example_2_path.read_text())

pd.DataFrame([
    {
        'supplier name': exc['supplier'].get('name'),
        'consumer location': exc['consumer'].get('location'),
        'value': exc['value'],
    }
    for exc in data['exchanges']
])


## 2) The main matching strategies in `edges`

The four main mapping strategies described in the questionnaire are complemented by a global fallback for remaining eligible exchanges.


In [ ]:
mapping_rules = pd.DataFrame([
    {'rule': 'direct mapping', 'example': 'FR -> FR', 'idea': 'The CF exists for the exact same location as the exchange.'},
    {'rule': 'aggregate-region mapping', 'example': 'ES -> RER', 'idea': 'A broader regional factor is used when no country-specific factor exists.'},
    {'rule': 'dynamic-location mapping', 'example': 'BR -> RoW', 'idea': 'A dynamic set such as RoW is constructed from the relevant included locations.'},
    {'rule': 'contained-location mapping', 'example': 'CA-QC -> CA', 'idea': 'A subregion inherits the factor of a containing location.'},
    {'rule': 'global fallback', 'example': 'XX -> GLO', 'idea': 'A global factor is used when no more specific match can be applied.'},
])
mapping_rules


## Checkpoint 1

Predict which mapping strategy should apply in each row.


In [ ]:
prediction_table = pd.DataFrame([
    {'inventory location': 'FR', 'available CF locations': 'FR, RER, GLO', 'predicted rule': ''},
    {'inventory location': 'ES', 'available CF locations': 'RER, GLO', 'predicted rule': ''},
    {'inventory location': 'BR', 'available CF locations': 'RoW, GLO', 'predicted rule': ''},
    {'inventory location': 'CA-QC', 'available CF locations': 'CA, GLO', 'predicted rule': ''},
    {'inventory location': 'XX', 'available CF locations': 'GLO', 'predicted rule': ''},
])
prediction_table


In [ ]:
prediction_table = pd.DataFrame([
    {'inventory location': 'FR', 'available CF locations': 'FR, RER, GLO', 'predicted rule': 'direct mapping'},
    {'inventory location': 'ES', 'available CF locations': 'RER, GLO', 'predicted rule': 'aggregate-region mapping'},
    {'inventory location': 'BR', 'available CF locations': 'RoW, GLO', 'predicted rule': 'dynamic-location mapping'},
    {'inventory location': 'CA-QC', 'available CF locations': 'CA, GLO', 'predicted rule': 'contained-location mapping'},
    {'inventory location': 'XX', 'available CF locations': 'GLO', 'predicted rule': 'global fallback'},
])
prediction_table


## 3) Which `edges` methods implement these steps?

The API mirrors the conceptual workflow quite closely.


In [ ]:
helper_methods = pd.DataFrame([
    {'method': 'map_exchanges()', 'role': 'Apply the direct matching rules encoded in the method definition.'},
    {'method': 'map_aggregate_locations()', 'role': 'Use broader static regions when exact matches are unavailable.'},
    {'method': 'map_dynamic_locations()', 'role': 'Handle dynamic geographies such as RoW.'},
    {'method': 'map_contained_locations()', 'role': 'Use factors for containing regions such as CA for CA-QC.'},
    {'method': 'map_remaining_locations_to_global()', 'role': 'Assign a global fallback to the remaining eligible exchanges.'},
    {'method': 'statistics()', 'role': 'Check how much of the inventory has been characterized after the mapping steps.'},
    {'method': 'generate_cf_table()', 'role': 'Inspect the factors that were finally assigned to each exchange.'},
])
helper_methods


## 4) Little tests on special geographies

We now attach simple numerical factors to a toy method so that we can test not only **which rule** applies, but also **which CF value** should be selected.

The values below are chosen only for teaching purposes:

- `CH -> 1.0`
- `FR -> 2.0`
- `RER -> 3.0`
- `CA -> 4.0`
- `RoW -> 5.0`
- `GLO -> 6.0`


In [ ]:
from constructive_geometries import Geomatcher


def label(item):
    return item[1] if isinstance(item, tuple) else str(item)


def labels(items):
    return [label(item) for item in items]


geo = Geomatcher()

toy_factors = pd.DataFrame([
    {"CF location": "CH", "CF": 1.0, "comment": "direct country-specific factor"},
    {"CF location": "FR", "CF": 2.0, "comment": "direct country-specific factor"},
    {"CF location": "RER", "CF": 3.0, "comment": "aggregate European region"},
    {"CF location": "CA", "CF": 4.0, "comment": "country factor for Canadian subregions"},
    {"CF location": "RoW", "CF": 5.0, "comment": "dynamic rest-of-world factor"},
    {"CF location": "GLO", "CF": 6.0, "comment": "global fallback"},
])
toy_factors

The next cell uses `constructive_geometries` to test the **static** parts of the matching logic.

Notice that `RoW` is different: it is not a static geography in `Geomatcher`. In `edges`, `RoW` is built dynamically from the locations that are still unmatched after direct and aggregate-region matches have already been used.


In [ ]:
def is_within(location, region):
    try:
        parents = set(labels(geo.within(location, include_self=False, exclusive=False, biggest_first=False)))
        return region in parents
    except KeyError:
        return False


special_geo_checks = pd.DataFrame([
    {
        "inventory location": "ES",
        "in RER": is_within("ES", "RER"),
        "in CA": is_within("ES", "CA"),
        "already covered before RoW": is_within("ES", "RER") or is_within("ES", "CA") or "ES" in {"CH", "FR"},
        "expected chosen CF location": "RER",
    },
    {
        "inventory location": "IT",
        "in RER": is_within("IT", "RER"),
        "in CA": is_within("IT", "CA"),
        "already covered before RoW": is_within("IT", "RER") or is_within("IT", "CA") or "IT" in {"CH", "FR"},
        "expected chosen CF location": "RER",
    },
    {
        "inventory location": "CA-QC",
        "in RER": is_within("CA-QC", "RER"),
        "in CA": is_within("CA-QC", "CA"),
        "already covered before RoW": is_within("CA-QC", "RER") or is_within("CA-QC", "CA") or "CA-QC" in {"CH", "FR"},
        "expected chosen CF location": "CA",
    },
    {
        "inventory location": "BR",
        "in RER": is_within("BR", "RER"),
        "in CA": is_within("BR", "CA"),
        "already covered before RoW": is_within("BR", "RER") or is_within("BR", "CA") or "BR" in {"CH", "FR"},
        "expected chosen CF location": "RoW",
    },
    {
        "inventory location": "XX",
        "in RER": False,
        "in CA": False,
        "already covered before RoW": False,
        "expected chosen CF location": "GLO",
    },
])
special_geo_checks

## Checkpoint 2

Use the table above and fill in the chosen CF location and the numerical factor that should be applied.


In [ ]:
matching_test = pd.DataFrame([
    {"inventory location": "ES", "chosen CF location": "", "matched CF": ""},
    {"inventory location": "IT", "chosen CF location": "", "matched CF": ""},
    {"inventory location": "CA-QC", "chosen CF location": "", "matched CF": ""},
    {"inventory location": "BR", "chosen CF location": "", "matched CF": ""},
    {"inventory location": "XX", "chosen CF location": "", "matched CF": ""},
])
matching_test

In [ ]:
matching_test = pd.DataFrame([
    {
        "inventory location": "ES",
        "chosen CF location": "RER",
        "matched CF": 3.0,
    },
    {
        "inventory location": "IT",
        "chosen CF location": "RER",
        "matched CF": 3.0,
    },
    {
        "inventory location": "CA-QC",
        "chosen CF location": "CA",
        "matched CF": 4.0,
    },
    {
        "inventory location": "BR",
        "chosen CF location": "RoW",
        "matched CF": 5.0,
    },
    {
        "inventory location": "XX",
        "chosen CF location": "GLO",
        "matched CF": 6.0,
    },
])
matching_test

## Checkpoint 3

Write a short explanation for each special case.

Focus on the logic, not on memorizing the answer:

- Why does `ES` get the `RER` factor?
- Why does `CA-QC` get the `CA` factor?
- Why does `BR` end up in `RoW` instead of `GLO`?
- Why does `XX` fall back to `GLO`?


In [ ]:
explanations = pd.DataFrame([
    {"case": "ES -> RER", "your explanation": ""},
    {"case": "CA-QC -> CA", "your explanation": ""},
    {"case": "BR -> RoW", "your explanation": ""},
    {"case": "XX -> GLO", "your explanation": ""},
])
explanations

In [ ]:
explanations = pd.DataFrame([
    {
        "case": "ES -> RER",
        "your explanation": "ES is geographically contained in RER, so the aggregate European factor can be used.",
    },
    {
        "case": "CA-QC -> CA",
        "your explanation": "CA-QC is a subregion of CA, so the factor for the containing geography CA can be inherited.",
    },
    {
        "case": "BR -> RoW",
        "your explanation": "BR is not already covered by CH, FR, RER, or CA, so it remains in the dynamic rest-of-world set and gets the RoW factor.",
    },
    {
        "case": "XX -> GLO",
        "your explanation": "No direct, aggregate, dynamic, or contained geography match is available, so the remaining option is the global fallback.",
    },
])
explanations

## 5) Optional scaffold on a real activity

This cell prepares an `EdgeLCIA` object and prints the intended sequence of mapping calls, without trying to force a full Day 2 calculation in every environment.


In [ ]:
import bw2data as bd
from pathlib import Path
from edges import EdgeLCIA

example_2_path = Path('/Users/romain/GitHub/barcelona-regionalized-lcia/tutorials/DAY 2/assets/edges_examples/lcia_example_2.json')
db_name = next((name for name in sorted(bd.databases) if name.startswith('bafu')), None)

if db_name is None:
    print('No BAFU database found in the current project.')
else:
    db = bd.Database(db_name)
    activity = next(
        act for act in db
        if 'passenger car' in act['name'].lower()
        and 'operation' in act['name'].lower()
    )
    lcia = EdgeLCIA(
        demand={activity: 1},
        filepath=str(example_2_path),
    )
    print('Prepared EdgeLCIA for:', activity['name'])
    print('Mapping sequence to try next:')
    print('lcia.lci()')
    print('lcia.map_exchanges()')
    print('lcia.map_aggregate_locations()')
    print('lcia.map_dynamic_locations()')
    print('lcia.map_contained_locations()')
    print('lcia.map_remaining_locations_to_global()')
    print('lcia.statistics()')


## Recap

After this notebook, you should now be able to:

- explain the main geography-matching strategies used by `edges`
- predict how simple geography cases should be handled
- trace why special geographies such as `RER`, `RoW`, `CA`, and `GLO` are selected
- identify which `edges` helper method corresponds to each matching step
- distinguish method definition from matching logic
